In [1]:
# An ugly backup method for the data engineering process (as of 10/16/23) 
# which assembles a supertable with relevant associations for AS/ML/DS

In [ ]:
 /*1*/
CREATE OR REPLACE TABLE [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9 (
  STUDENT_ID NUMBER(38,0), 
  SESSION_ID VARCHAR(16777216), CREATED_AT TIMESTAMP_TZ(9), 
  PRODUCT VARCHAR(16777216), MATH_QUESTION_ID VARCHAR(16777216),
  TRIAL_ID VARCHAR(16777216),
  RL_TOP_LEVEL_SKILL_ID VARCHAR(16777216), PRACTICED_RL_SKILL_ID VARCHAR(16777216), 
  ANSWERS_RL_TOP_LEVEL_SKILL_ID VARCHAR(16777216), ANSWERS_PRACTICED_RL_SKILL_ID VARCHAR(16777216), 
  SKILL_USP_ID VARCHAR(16777216), 
  DURATION_SECONDS INT, CORRECTNESS FLOAT, ATTEMPTS INT, 
  STUDENT_STANDARD_SET_ID VARCHAR(16777216), STANDARD_NAME VARCHAR(16777216), 
  SKILL_NAME VARCHAR(16777216), SKILL_PROGRESSION_ORDER VARCHAR(16777216), 
  SKILL_GRADE FLOAT, STANDARD_POSITION_IN_GRADE_DOMAIN INT,
  STANDARD_POSITION_IN_DOMAIN INT,
  STANDARDS_IN_GRADE_DOMAIN INT,
  STANDARDS_IN_DOMAIN INT,STANDARD_GRADE_LEVEL FLOAT, 
  SKILL_REMOVED BOOLEAN, SKILL_IS_FOCUS BOOLEAN,
  DOMAIN_ID VARCHAR(16777216),DOMAIN_NAME VARCHAR(16777216),
  SUPER_SKILL_ORDER INT,

  ASSIGNMENT_ID VARCHAR(16777216), SESSION_ACCURACY NUMBER, 
  SELF_ASSIGNED BOOLEAN, PRACTICE BOOLEAN, PRETEST BOOLEAN, CHALLENGE BOOLEAN,

  TRIAL_NUMBER INT,
  SKILL_DIFF INT,
  SESSION_TRIAL_NUMBER INT,
  IN_REMEDIATION BOOLEAN
 );

/*2*/
/*--Creates table with a bunch of JOINS to find student standard set, and figure out the 
--super skill ordering at a trial level
-- ~26 hours.... took 4+ hours now -- TOOK 2.5 NOW
--SELECT * FROM [REDACTED].views.math_answers_fact LIMIT 10;*/

INSERT INTO [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9 (
  
select 
STUDENT_ID, SESSION_ID, CREATED_AT, PRODUCT, MATH_QUESTION_ID, TRIAL_ID, 
RL_TOP_LEVEL_SKILL_ID, PRACTICED_RL_SKILL_ID, ANSWERS_RL_TOP_LEVEL_SKILL_ID, 
ANSWERS_PRACTICED_RL_SKILL_ID, SKILL_USP_ID, DURATION_SECONDS, CORRECTNESS, ATTEMPTS, 
STUDENT_STANDARD_SET_ID, STANDARD_NAME, SKILL_NAME, SKILL_PROGRESSION_ORDER, 
SKILL_GRADE, STANDARD_POSITION_IN_GRADE_DOMAIN,STANDARD_POSITION_IN_DOMAIN,
STANDARDS_IN_GRADE_DOMAIN,STANDARDS_IN_DOMAIN,STANDARD_GRADE_LEVEL, 
SKILL_REMOVED, SKILL_IS_FOCUS,DOMAIN_ID,DOMAIN_NAME,SUPER_SKILL_ORDER,
assignment_id, CAST(accuracy as NUMBER) as SESSION_ACCURACY, self_assigned, PRACTICE, PRETEST, CHALLENGE,
--create skill diff column
TRIAL_NUMBER,
SUPER_SKILL_ORDER-LAG(SUPER_SKILL_ORDER,1) 
OVER(PARTITION BY STUDENT_ID, DOMAIN_NAME order by TRIAL_NUMBER ASC) as SKILL_DIFF,
ROW_NUMBER() OVER(PARTITION BY SESSION_ID order by TRIAL_NUMBER ASC) as SESSION_TRIAL_NUMBER,
CASE WHEN RL_TOP_LEVEL_SKILL_ID = PRACTICED_RL_SKILL_ID THEN False
  WHEN  PRACTICED_RL_SKILL_ID is NULL THEN False
  ELSE True END as IN_REMEDIATION

 from
 (select *,
 ROW_NUMBER() OVER (PARTITION BY STUDENT_ID, DOMAIN_NAME ORDER BY TRIAL_ID ASC) as TRIAL_NUMBER 
  from
(

  select * from

( select assignment_session_id, created_at, product, maf1.math_question_id, answers_rl_top_level_skill_id,
  answers_practiced_rl_skill_id, duration_seconds, correctness, attempts, answer, trial_id,
  student_id, session_id, rl_top_level_skill_id, practiced_rl_skill_id
 
 from
   (select assignment_session_id, created_at, 'adaptive' as product, math_question_id, rl_skill_id as answers_rl_top_level_skill_id,
  COALESCE(rl_sub_skill_id, rl_skill_id) as answers_practiced_rl_skill_id, duration_seconds, correctness, attempts, answer, id as trial_id
  from [REDACTED].prod.answers) fp1
  LEFT JOIN
  (select  session_id, math_question_id, student_id, rl_top_level_skill_id, practiced_rl_skill_id  from
  [REDACTED].views.math_answers_fact) maf1
  on maf1.session_id = fp1.assignment_session_id
  and maf1.math_question_id = fp1.math_question_id
 
    //and CREATED_AT >= DATEADD(year, -2, GETDATE())
    //and CREATED_AT <= DATEADD(year, -1, GETDATE())
) MAF

/*join student_id to teacher_id*/
LEFT JOIN
(select distinct(student_id) as sbtd_student_id, student_standard_set_id 
 from [REDACTED].views.students_by_teacher_dim
) sbtd on sbtd_student_id = MAF.student_id

/*join teacher_id to standard_set*/
/* could use but don't need becasue standard_set_id is in students_by_teacher_dim table */

/*join standard_set to skill_ordering*/
LEFT JOIN
/*bunch of DISTINCT ON alternative SQL for snowflake to get rid of duplicates while selecting all*/
(SELECT * FROM [REDACTED].views.rl_skills_dim 
 QUALIFY ROW_NUMBER() OVER (PARTITION BY standard_set_id,domain_id,skill_usp_id ORDER BY standard_set_id,domain_id,skill_usp_id) = 1) rsd
  
  on rsd.standard_set_id = sbtd.student_standard_set_id and MAF.answers_practiced_rl_skill_id = rsd.skill_usp_id
) annotated_maf

LEFT JOIN --fetches teacher assigned standard
(select distinct(student_id) as ts_student_id, student_standard_set_id as teacher_assigned_standard 
 from [REDACTED].views.students_by_teacher_dim
) teacher_standards
on annotated_maf.student_id = teacher_standards.ts_student_id


LEFT JOIN --This grabs the data we need to LATER generate a superordering of the skills
(
  select
    distinct(skill_usp_id) as skill_id, -- For some unknown reasons we have duplicates
    skill_progression_order as skill_progression_order_2,
    standard_position_in_domain as standard_position_in_domain_2,
    domain_name as skill_domain_name,
    standard_set_id
  from
    views.rl_skills_dim
) skill_info
ON skill_info.skill_id = annotated_maf.skill_usp_id
and skill_info.standard_set_id = student_standard_set_id

LEFT JOIN --Need to find the Superordering of skills 
(
  --generates a super ordering of skills
  select skill_usp_id as skill_id_3, 
  --FIRST GROUP BY domain and StandardSET, then order by domain and skill progression
  row_number() OVER(PARTITION by standard_set_id,domain_name order by
      standard_position_in_domain ASC,
      skill_progression_order ASC) as SUPER_SKILL_ORDER,
      standard_set_id, domain_name as domain_name_3
  FROM /*need distinct filtering bc there's duplicates in skill table*/
      (select distinct standard_set_id, domain_name, STANDARD_POSITION_IN_DOMAIN, SKILL_PROGRESSION_ORDER, skill_usp_id from views.rl_skills_dim)
) superskillorder
--Join each item on its teacher assigned standard_set, skill_id
ON superskillorder.skill_id_3 = skill_id
and superskillorder.standard_set_id = teacher_assigned_standard
--add labels for pretest/practice so we can include/exclude these -- throwing accuracy in there for a double-check
LEFT JOIN
(select id, assignment_id, accuracy from prod.math_adaptive_assignment_sessions
) sessions
on session_id = sessions.id

LEFT join 
(select id as id2, self_assigned, type='practice' as PRACTICE, type='pretest' as PRETEST, type='challenge' as CHALLENGE
      from prod.math_adaptive_assignments
) assignments
  on assignment_id=assignments.id2


) 
);

/*3*/
/*--Add placement events--30 min*/

ALTER TABLE [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9
ADD COLUMN most_recent_placement_event DATE, placement_type VARCHAR(16777216);


/*4*/
UPDATE [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9 AS ay
SET most_recent_placement_event = (
    SELECT MAX(mdp.completed_at)
    FROM [REDACTED].prod.math_domain_placements AS mdp
    WHERE ay.student_id = mdp.student_id AND ay.created_at > mdp.completed_at
);

/*4b*/
UPDATE [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9 AS ay
SET placement_type = (
    SELECT MAX(mdp.source)  /*a windowing func to select one val only, without using ROW() bc its unsupported*/
    FROM [REDACTED].prod.math_domain_placements AS mdp
    WHERE ay.student_id = mdp.student_id AND ay.most_recent_placement_event = TO_DATE(mdp.completed_at) 
  /*date in prev step gets converted to YYYYMMDD from YYYYMMDD with a timestamp*/
);

/*4c needed to add star score for placement & star-to-performance analyses*/
ALTER TABLE [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9
ADD COLUMN STAR_SCORE NUMBER(38,0);
/*4d*/
UPDATE [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9 AS ay
SET star_score = (
    SELECT MAX(mdp.star_score)  /*a windowing func to select one val only, without using ROW() bc its unsupported*/
    FROM [REDACTED].prod.math_domain_placements AS mdp
    WHERE ay.student_id = mdp.student_id AND ay.most_recent_placement_event = TO_DATE(mdp.completed_at) 
  /*date in prev step gets converted to YYYYMMDD from YYYYMMDD with a timestamp*/
);



/*5*/
ALTER TABLE [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9
ADD COLUMN placement_trial_number INT,  all_placement_trial_number INT, STARTING_SUPER_SKILL_ORDER INT;


/*6*/
/*add trial # since last placcement (that only counts core algo trials), all_trial_# that includes all trials since last palcement,
and starting skill -- so we dont keep tracking current skill when evaluating placement quality */

CREATE TEMPORARY TABLE temp_placement_tn AS
WITH placement_cte AS (
    SELECT
        student_id,
        domain_id,
        trial_number,
        ROW_NUMBER() OVER (PARTITION BY student_id, domain_id, grp ORDER BY trial_number) AS placement_TN,
        FIRST_VALUE(SUPER_SKILL_ORDER) OVER (PARTITION BY student_id, domain_id, grp ORDER BY trial_number) as STARTING_SUPER_SKILL_ORDER
    FROM (
        SELECT
            student_id,
            domain_id,
            trial_number,
            SUM(is_placement_change) OVER (PARTITION BY student_id, domain_id ORDER BY trial_number) AS grp,
            SUPER_SKILL_ORDER
        FROM (
            SELECT
                student_id,
                domain_id,
                trial_number,
                SUPER_SKILL_ORDER,
                CASE
                    WHEN most_recent_placement_event IS NULL THEN 0
                    WHEN EQUAL_NULL(LAG(most_recent_placement_event, 1) OVER (PARTITION BY student_id, domain_id ORDER BY trial_number), 
                                    most_recent_placement_event) THEN 0
                    ELSE 1
                END AS is_placement_change
            FROM [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9
            WHERE IN_REMEDIATION = FALSE AND SESSION_TRIAL_NUMBER != 1 AND SESSION_TRIAL_NUMBER != 2 AND PRACTICE=TRUE
            ORDER BY domain_id, trial_number
        )
    ) AS subquery WHERE grp>0 /*exclude no placement trials bc they might be pretest */
)
SELECT student_id, domain_id, trial_number, placement_TN, STARTING_SUPER_SKILL_ORDER
FROM placement_cte;


/*7*/
/*-- Step 2: Update the main table using the temporary table, 30min*/
UPDATE [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9 AS ay
SET placement_trial_number = temp.placement_TN,
    STARTING_SUPER_SKILL_ORDER = temp.STARTING_SUPER_SKILL_ORDER
FROM temp_placement_tn AS temp
WHERE ay.student_id = temp.student_id
  AND ay.domain_id = temp.domain_id
  AND ay.trial_number = temp.trial_number;

/*8*/
/* Step 3: Create temporary table with placement_TN values FOR ALL
--6 min */
CREATE TEMPORARY TABLE temp_placement_tn_2 AS
WITH placement_cte AS (
    SELECT
        student_id,
        domain_id,
        trial_number,
        ROW_NUMBER() OVER (PARTITION BY student_id, domain_id, grp ORDER BY trial_number) AS placement_TN
    FROM (
        SELECT
            student_id,
            domain_id,
            trial_number,
            SUM(is_placement_change) OVER (PARTITION BY student_id, domain_id ORDER BY trial_number) AS grp
        FROM (
            SELECT
                student_id,
                domain_id,
                trial_number,
                CASE
                    WHEN most_recent_placement_event IS NULL THEN 0
                    WHEN EQUAL_NULL(LAG(most_recent_placement_event, 1) OVER (PARTITION BY student_id, domain_id ORDER BY trial_number), 
                                    most_recent_placement_event) THEN 0
                    ELSE 1
                END AS is_placement_change
            FROM [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9
            ORDER BY domain_id, trial_number
        )
    )  AS subquery WHERE grp>0 /*exclude no placement trials bc they might be pretest */
)
SELECT student_id, domain_id, trial_number, placement_TN
FROM placement_cte;

/*9*/
/*-- Step 4: Update the main table using the temporary table, 30min*/
UPDATE [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9 AS ay
SET all_placement_trial_number = temp.placement_TN
FROM temp_placement_tn_2 AS temp
WHERE ay.student_id = temp.student_id
  AND ay.domain_id = temp.domain_id
  AND ay.trial_number = temp.trial_number;
 
/*10*/

/*select * from [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9 where placement_trial_number is not null limit 100;
select * from [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9 where student_id = 28850612;
select * from temp_placement_tn_2 limit 10;
*/

/*12*/
/* add another column for labelling Remediation_entry and exit events 
and providing an updated SKILL_MOVEMENT that excludes review trials and REMEDIATION and PRETEST
-- 1.5 hours*/
ALTER TABLE [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9
ADD COLUMN UPDATED_SKILL_DIFF INT, REMEDIATION_ENTRY BOOLEAN;

/*13*/
/*This calculates a 'True' Skill Movement, basically by ignoring Remediation trials and first 2 trials (SR trials)
*/ 
MERGE INTO [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9 AS t
USING (
  SELECT student_id,
    domain_name,
    trial_number,
  
  CASE WHEN LAG(IN_REMEDIATION) OVER(PARTITION BY STUDENT_ID, DOMAIN_NAME order by TRIAL_NUMBER ASC)
  =False and IN_REMEDIATION=True 
  THEN True
    WHEN LAG(IN_REMEDIATION) OVER(PARTITION BY STUDENT_ID, DOMAIN_NAME order by TRIAL_NUMBER ASC)
  =True and IN_REMEDIATION=False 
  THEN False
  Else Null End
  as new_REMEDIATION_ENTRY
  FROM (select * from [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9
  WHERE SESSION_TRIAL_NUMBER != 1 and SESSION_TRIAL_NUMBER != 2 
        and PRACTICE=True)
  
) AS subquery
ON t.student_id = subquery.student_id
  AND t.domain_name = subquery.domain_name
  AND t.trial_number = subquery.trial_number
WHEN MATCHED THEN
  UPDATE SET t.REMEDIATION_ENTRY=subquery.new_REMEDIATION_ENTRY;


/*14 -- calcs Skill movement */
MERGE INTO [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9 AS t
USING (
  SELECT student_id,
    domain_name,
    trial_number,
  SUPER_SKILL_ORDER-LAG(SUPER_SKILL_ORDER,1) 
    OVER(PARTITION BY STUDENT_ID, DOMAIN_NAME order by TRIAL_NUMBER ASC) as new_UPDATED_SKILL_DIFF
  FROM (select * from [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9
  WHERE IN_REMEDIATION != TRUE and SESSION_TRIAL_NUMBER != 1 and SESSION_TRIAL_NUMBER != 2 
        and PRACTICE=True)
  
) AS subquery
ON t.student_id = subquery.student_id
  AND t.domain_name = subquery.domain_name
  AND t.trial_number = subquery.trial_number
WHEN MATCHED THEN
  UPDATE SET t.UPDATED_SKILL_DIFF = subquery.new_UPDATED_SKILL_DIFF;



/*15
--type is messed up
ALTER TABLE [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9
ADD COLUMN SESSION_ACCURACY NUMBER;

/*16
UPDATE [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9
SET session_accuracy = CAST(accuracy AS NUMBER);

select * from [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9 limit 10;
*/


/*ADD markers when they just exited PRETEST*/
ALTER TABLE [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9
ADD COLUMN PRETEST_START BOOLEAN;

CREATE OR REPLACE TEMPORARY TABLE temp_cte AS (
  SELECT
    student_id,
    domain_id,
    trial_number,
    PRETEST,
    LAG(PRETEST) OVER (PARTITION BY student_id, domain_id ORDER BY trial_number) AS prev_pretest
  FROM [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9
);

UPDATE [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9 AS ay
SET PRETEST_START = (
  CASE WHEN ay.PRETEST = FALSE AND temp_cte.prev_pretest = TRUE
  THEN TRUE
  ELSE FALSE
  END
)
FROM temp_cte
WHERE
  ay.student_id = temp_cte.student_id AND
  ay.domain_id = temp_cte.domain_id AND
  ay.trial_number = temp_cte.trial_number;




/*now add trial numbers based on last pretest*/
/*17*/
ALTER TABLE [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9
ADD COLUMN pretest_trial_number INT;

/*17b*/
/*-- Step 1: Create temporary table with pretest_TN values
--6 min*/
CREATE OR REPLACE TEMPORARY TABLE temp_pretest_tn AS
WITH pretest_cte AS (
    SELECT
        student_id,
        domain_id,
        trial_number,
        ROW_NUMBER() OVER (PARTITION BY student_id, domain_id, grp ORDER BY trial_number) AS pretest_TN
    FROM (
        SELECT
            student_id,
            domain_id,
            trial_number,
            SUM(is_pretest_change) OVER (PARTITION BY student_id, domain_id ORDER BY trial_number) AS grp
        FROM (
            SELECT
                student_id,
                domain_id,
                trial_number,
                CASE WHEN pretest_start=TRUE THEN 1 Else 0 END as is_pretest_change
            FROM [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9
            WHERE IN_REMEDIATION = FALSE AND SESSION_TRIAL_NUMBER != 1 AND SESSION_TRIAL_NUMBER != 2 AND PRACTICE=TRUE
            ORDER BY domain_id, trial_number
        )
    ) AS subquery
)
SELECT student_id, domain_id, trial_number, pretest_TN
FROM pretest_cte;

/*17c*/
/*-- Step 2: Update the main table using the temporary table, 30min*/
UPDATE [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9 AS ay
SET pretest_trial_number = temp.pretest_TN
FROM temp_pretest_tn AS temp
WHERE ay.student_id = temp.student_id
  AND ay.domain_id = temp.domain_id
  AND ay.trial_number = temp.trial_number;




/*now add trial numbers based on last pretest AND by SKILL_ID (using super_skill_order)*/
/*this lets us fetch the first X trials for a skill after a placement*/

/*18a*/
ALTER TABLE [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9
ADD COLUMN pretest_skill_trial_number INT;

/*18b*/
/*-- Step 1: Create temporary table with pretest_TN values
--6 min*/
CREATE OR REPLACE TEMPORARY TABLE temp_pretest_tn AS
WITH pretest_cte AS (
    SELECT
        student_id,
        domain_id,
        trial_number,
        ROW_NUMBER() OVER (PARTITION BY student_id, domain_id, super_skill_order, grp ORDER BY trial_number) AS pretest_SKILL_TN
    FROM (
        SELECT
            student_id,
            domain_id,
            super_skill_order,
            trial_number,
            SUM(is_pretest_change) OVER (PARTITION BY student_id, domain_id ORDER BY trial_number) AS grp
        FROM (
            SELECT
                student_id,
                domain_id,
                super_skill_order,
                trial_number,
                CASE WHEN pretest_start=TRUE THEN 1 Else 0 END as is_pretest_change
            FROM [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9
            WHERE IN_REMEDIATION = FALSE AND SESSION_TRIAL_NUMBER != 1 AND SESSION_TRIAL_NUMBER != 2 AND PRACTICE=TRUE
            ORDER BY domain_id, trial_number
        )
    ) AS subquery
)
SELECT student_id, domain_id, trial_number, pretest_SKILL_TN
FROM temp_pretest_tn;

/*18c*/
/*-- Step 2: Update the main table using the temporary table, 30min*/
UPDATE [REDACTED].product_testing.annotated_skill_stretches_math_all_years_v9 AS ay
SET pretest_skill_trial_number = temp.pretest_SKILL_TN
FROM temp_pretest_tn AS temp
WHERE ay.student_id = temp.student_id
  AND ay.domain_id = temp.domain_id
  AND ay.trial_number = temp.trial_number;

